# Agent Memory Toolkit – Demo

This notebook walks through the **Agent Memory Toolkit** library using the synchronous `CosmosMemoryClient` class:

1. **Setup** – Install dependencies and load environment variables
2. **Local memory operations** – `add_local`, `get_local`, `update_local`, `delete_local`
3. **Cosmos DB operations** – `add_cosmos`, `get_memories`, `get_thread`
4. **Thread Summary** – `generate_thread_summary()` (in-process LLM)
5. **Memory Extraction** – `extract_memories()` (facts + episodic + procedural)
6. **User Summary** – `generate_user_summary()` (cross-thread profile)
7. **Vector / hybrid search** – `search_cosmos()`
8. **Tagging, salience & deduplication** – tag mutation, salience filter, `reconcile()`
9. **Automatic processing (Change Feed)** – optional Azure Function for background processing

> 💡 **Tip:** the synchronous `CosmosMemoryClient` accepts an optional `processor=` kwarg (defaults to `InProcessProcessor`). Pass `DurableFunctionProcessor()` to delegate summarization to the sibling Azure Function app — see `Samples/scenario_remote_processor.py`.


## 1. Setup

Install/import dependencies and load environment variables from `.env`.

In [1]:
import os, json

from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential



# Add parent directory to path so we can import the package easily

import sys

sys.path.insert(0, os.path.abspath(".."))



from agent_memory_toolkit import CosmosMemoryClient



# Load environment variables from .env in the repo root

load_dotenv(os.path.join("..", ".env"))



print("COSMOS_DB_ENDPOINT:", os.getenv("COSMOS_DB_ENDPOINT"))

print("COSMOS_DB_DATABASE:", os.getenv("COSMOS_DB_DATABASE"))

print("COSMOS_DB_CONTAINER:", os.getenv("COSMOS_DB_CONTAINER"))

print("COSMOS_DB_COUNTERS_CONTAINER:", os.getenv("COSMOS_DB_COUNTERS_CONTAINER", "counter"))

print("COSMOS_DB_LEASE_CONTAINER:", os.getenv("COSMOS_DB_LEASE_CONTAINER", "leases"))

print("COSMOS_DB_THROUGHPUT_MODE:", os.getenv("COSMOS_DB_THROUGHPUT_MODE", "serverless"))

print("COSMOS_DB_AUTOSCALE_MAX_RU:", os.getenv("COSMOS_DB_AUTOSCALE_MAX_RU", "1000"))

COSMOS_DB_ENDPOINT: https://akataria-agent-memory-testing.documents.azure.com:443/
COSMOS_DB_DATABASE: ai_memory
COSMOS_DB_CONTAINER: memories
COSMOS_DB_COUNTERS_CONTAINER: counter
COSMOS_DB_LEASE_CONTAINER: leases
COSMOS_DB_THROUGHPUT_MODE: serverless
COSMOS_DB_AUTOSCALE_MAX_RU: 1000


In [2]:
# Create a CosmosMemoryClient instance.
# Credential priority: explicit cosmos_credential > explicit cosmos_key > DefaultAzureCredential.
# Set COSMOS_DB_KEY in your .env if you don't yet have control-plane RBAC (currently in private preview).
memory = CosmosMemoryClient(
    cosmos_endpoint=os.getenv("COSMOS_DB_ENDPOINT"),
    cosmos_database=os.getenv("COSMOS_DB_DATABASE"),
    cosmos_container=os.getenv("COSMOS_DB_CONTAINER"),
    cosmos_counter_container=os.getenv("COSMOS_DB_COUNTERS_CONTAINER", "counter"),
    cosmos_lease_container=os.getenv("COSMOS_DB_LEASE_CONTAINER", "leases"),
    cosmos_throughput_mode=os.getenv("COSMOS_DB_THROUGHPUT_MODE", "serverless"),
    cosmos_autoscale_max_ru=int(os.getenv("COSMOS_DB_AUTOSCALE_MAX_RU", "1000")),
    cosmos_key=os.getenv("COSMOS_DB_KEY"),
    ai_foundry_endpoint=os.getenv("AI_FOUNDRY_ENDPOINT"),
    ai_foundry_api_key=os.getenv("AI_FOUNDRY_API_KEY"),
    embedding_deployment_name=os.getenv("AI_FOUNDRY_EMBEDDING_DEPLOYMENT_NAME", "text-embedding-3-large"),
    chat_deployment_name=os.getenv("AI_FOUNDRY_CHAT_DEPLOYMENT_NAME", "gpt-4o-mini"),
    use_default_credential=True,
)

print("CosmosMemoryClient instance created")
print("Throughput mode:", os.getenv("COSMOS_DB_THROUGHPUT_MODE", "serverless"))
print("Local memory store:", memory.local_memory)


CosmosMemoryClient instance created
Throughput mode: serverless
Local memory store: []


## 2. Local Memory Operations

### 2a. Add memories with `add_local`

Each memory has a `user_id`, `role`, `content`, optional `type` (raw/summary/fact), and optional `metadata`.

In [3]:
import uuid
THREAD_ID = str(uuid.uuid4())
# Use a unique user_id per demo run so we get a clean extraction without
# inheriting facts from prior runs that would dedup new content away.
USER_ID = f"user-{uuid.uuid4().hex[:8]}"
print(f"User ID:   {USER_ID}")
print(f"Thread ID: {THREAD_ID}\n")


User ID:   user-09577899
Thread ID: e14d2b93-dbdc-46e7-99db-d3c916198047



In [4]:
# Add sample conversation: weather in Seattle → booking a trip (6 turns)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="What's the weather like in Seattle this weekend?",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="This weekend Seattle will be around 55°F with partly cloudy skies on Saturday and light rain expected Sunday.",
)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="That sounds nice enough. Can you help me book a trip to Seattle for this weekend?",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="Sure! I found round-trip flights departing Friday evening and returning Sunday night. There are also several hotels in downtown Seattle with availability. Would you like me to look at specific airlines or neighborhoods?",
)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="Something near Pike Place Market would be great. And keep the flights under $300 round trip if possible.",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="I found a round-trip on Alaska Airlines for $275 and two hotel options within a 5-minute walk of Pike Place Market: the Inn at the Market ($189/night) and a Hilton Garden Inn ($145/night). Want me to reserve one of these?",
)

memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="Whenever you book a flight for me, always book an aisle seat — never a window or middle.",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="Got it. I'll always select an aisle seat for your bookings.",
)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="For trip planning, my workflow is: first check the weather for the destination, then check flights, then book the hotel last after everything else is confirmed.",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="Noted — I'll follow that order: weather, then flights, then hotel.",
)
memory.add_local(
    user_id=USER_ID, role="user", thread_id=THREAD_ID,
    content="And never book me into anything that departs or arrives between midnight and 6am unless I explicitly approve it.",
)
memory.add_local(
    user_id=USER_ID, role="agent", thread_id=THREAD_ID,
    content="Will do — no overnight bookings without your explicit approval.",
)

print(f"Added {len(memory.local_memory)} memories")
print(json.dumps(memory.get_local(), indent=2))

# A second short thread of pure procedural-style instructions. Demonstrates
# that the extractor produces clean procedural items when the conversation is
# focused on rules/workflows rather than mixed with factual booking specifics.
RULES_THREAD_ID = str(uuid.uuid4())
for role, content in [
    ("user",  "Whenever you book a flight for me, always book an aisle seat — never a window or middle."),
    ("agent", "Got it. I'll always select an aisle seat for your bookings."),
    ("user",  "For trip planning, my workflow is: first check the weather, then check flights, and book the hotel last after everything else is confirmed."),
    ("agent", "Noted — I'll follow that order: weather, then flights, then hotel."),
    ("user",  "Never book me into anything that departs or arrives between midnight and 6am unless I explicitly approve it."),
    ("agent", "Will do — no overnight bookings without your explicit approval."),
    ("user",  "When picking a hotel, only recommend ones that include complimentary breakfast."),
    ("agent", "Understood — only hotels with complimentary breakfast."),
]:
    memory.add_local(user_id=USER_ID, role=role, thread_id=RULES_THREAD_ID, content=content)

print(f"Rules thread ID: {RULES_THREAD_ID} ({sum(1 for m in memory.local_memory if m['thread_id']==RULES_THREAD_ID)} turns)")


Added 12 memories
[
  {
    "id": "a54fd759-3890-4c76-a477-f34c1e1ef6da",
    "user_id": "user-09577899",
    "thread_id": "e14d2b93-dbdc-46e7-99db-d3c916198047",
    "role": "user",
    "type": "turn",
    "content": "What's the weather like in Seattle this weekend?",
    "metadata": {},
    "created_at": "2026-05-04T20:26:45.942713+00:00",
    "tags": [],
    "ttl": 2592000
  },
  {
    "id": "34f80709-d595-4a2d-b725-ba0b2fbed14e",
    "user_id": "user-09577899",
    "thread_id": "e14d2b93-dbdc-46e7-99db-d3c916198047",
    "role": "agent",
    "type": "turn",
    "content": "This weekend Seattle will be around 55\u00b0F with partly cloudy skies on Saturday and light rain expected Sunday.",
    "metadata": {},
    "created_at": "2026-05-04T20:26:45.942755+00:00",
    "tags": [],
    "ttl": 2592000
  },
  {
    "id": "53301627-d160-47bd-bcdd-eff5901ac8d7",
    "user_id": "user-09577899",
    "thread_id": "e14d2b93-dbdc-46e7-99db-d3c916198047",
    "role": "user",
    "type": "turn",
  

### 2b. Query memories with `get_local`

Retrieve all memories, or filter by `memory_id`, `user_id`, `role`, or `memory_type`. Filters are combined with AND logic.

In [5]:
# Get all memories
all_memories = memory.get_local()
print(f"Total memories: {len(all_memories)}\n")

# Filter by user_id
user1_memories = memory.get_local(user_id=USER_ID)
print(f"Memories for user-001: {len(user1_memories)}")

# Filter by role
tool_memories = memory.get_local(role="tool")
print(f"Tool memories: {len(tool_memories)}")
for m in tool_memories:
    print(f"  [{m['id'][:8]}...] {m['content'][:60]}")

# Filter by type
facts = memory.get_local(memory_type="fact")
print(f"\nFact memories: {len(facts)}")
for m in facts:
    print(f"  [{m['id'][:8]}...] {m['content']}")

# Combine filters: user-001 + agent role
user1_agent = memory.get_local(user_id=USER_ID, role="agent")
print(f"\nAgent memories for user-001: {len(user1_agent)}")

Total memories: 20

Memories for user-001: 20
Tool memories: 0

Fact memories: 0

Agent memories for user-001: 10


### 2c. Update a memory with `update_local`

Update any combination of `content`, `role`, `memory_type`, or `metadata` for an existing memory by its `id`.

In [6]:
# Update the user's budget constraint to be more specific
target_id = memory.local_memory[4]["id"]  # "Something near Pike Place Market..."
print(f"Before update:\n{json.dumps(memory.get_local(memory_id=target_id)[0], indent=2)}\n")

memory.update_local(
    memory_id=target_id,
    content="Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
    metadata={"edited": True, "reason": "user clarified hotel budget"},
)
print(f"After update:\n{json.dumps(memory.get_local(memory_id=target_id)[0], indent=2)}")

Before update:
{
  "id": "131015e6-1239-45ec-aec8-c4296bd62146",
  "user_id": "user-09577899",
  "thread_id": "e14d2b93-dbdc-46e7-99db-d3c916198047",
  "role": "user",
  "type": "turn",
  "content": "Something near Pike Place Market would be great. And keep the flights under $300 round trip if possible.",
  "metadata": {},
  "created_at": "2026-05-04T20:26:45.942831+00:00",
  "tags": [],
  "ttl": 2592000
}

After update:
{
  "id": "131015e6-1239-45ec-aec8-c4296bd62146",
  "user_id": "user-09577899",
  "thread_id": "e14d2b93-dbdc-46e7-99db-d3c916198047",
  "role": "user",
  "type": "turn",
  "content": "Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
  "metadata": {
    "edited": true,
    "reason": "user clarified hotel budget"
  },
  "created_at": "2026-05-04T20:26:45.942831+00:00",
  "tags": [],
  "ttl": 2592000,
  "updated_at": "2026-05-04T20:26:45.951502+00:00"
}


### 2d. Delete a memory with `delete_local`

Remove a memory by its `id`.

In [7]:
# Delete the tool memory (index 2 – the tool call)
tool_memory_id = memory.local_memory[2]["id"]
print(f"Deleting memory {tool_memory_id[:8]}...")
memory.delete_local(tool_memory_id)

# Verify it's gone
print(f"\nRemaining memories: {len(memory.get_local())}")
for m in memory.get_local():
    print(f" [{m['thread_id'][:8]}...]  [{m['id'][:8]}...] role={m['role']:<6} type={m['type']:<8} {m['content'][:50]}")

Deleting memory 53301627...

Remaining memories: 19
 [e14d2b93...]  [a54fd759...] role=user   type=turn     What's the weather like in Seattle this weekend?
 [e14d2b93...]  [34f80709...] role=agent  type=turn     This weekend Seattle will be around 55°F with part
 [e14d2b93...]  [8a47c8f7...] role=agent  type=turn     Sure! I found round-trip flights departing Friday 
 [e14d2b93...]  [131015e6...] role=user   type=turn     Something near Pike Place Market would be great. K
 [e14d2b93...]  [f01ae2bb...] role=agent  type=turn     I found a round-trip on Alaska Airlines for $275 a
 [e14d2b93...]  [fd18c1e3...] role=user   type=turn     Whenever you book a flight for me, always book an 
 [e14d2b93...]  [b47560e8...] role=agent  type=turn     Got it. I'll always select an aisle seat for your 
 [e14d2b93...]  [a1c02435...] role=user   type=turn     For trip planning, my workflow is: first check the
 [e14d2b93...]  [fe6666fd...] role=agent  type=turn     Noted — I'll follow that order: weathe

## 3. Cosmos DB Operations

### 3a. Cosmos DB Connection

The client auto-connects to Cosmos DB when `cosmos_endpoint` is provided in the constructor. You can also call `connect_cosmos()` explicitly to reconnect or override connection parameters.

> **Prerequisites:**
> - A Cosmos DB for NoSQL account with a database and container matching your `.env` values
> - The container should have a [vector embedding policy](https://learn.microsoft.com/azure/cosmos-db/nosql/vector-search) configured on the `embedding` field
> - Entra ID / managed identity RBAC role (e.g. *Cosmos DB Built-in Data Contributor*)
> - An Azure AI Foundry embedding model deployment for `add_cosmos` and `search_cosmos`

In [8]:
# Already connected via constructor — call connect_cosmos() only if you need to reconnect
print(f"Connected: {memory._container_client is not None}")

Connected: True


### 3b. Add memories to Cosmos DB with `add_cosmos`

In [9]:
memory.push_to_cosmos()

In [10]:
# Push a new thread directly to Cosmos DB without adding to local memory first
new_thread_id = str(uuid.uuid4())
print(f"New Thread ID: {new_thread_id}\n")

# Add memories directly to Cosmos DB using add_cosmos
memory.add_cosmos(
    user_id="user-002", role="user", thread_id=new_thread_id,
    content="Can you recommend some good restaurants in New York City?",
)
memory.add_cosmos(
    user_id="user-002", role="tool", thread_id=new_thread_id,
    content='{"query": "top restaurants NYC", "results": ["Carbone", "Nobu", "Katz\'s Deli", "Le Bernardin"]}',
    metadata={"tool_name": "restaurant_search", "tool_call_id": "call_abc123"},
)
memory.add_cosmos(
    user_id="user-002", role="agent", thread_id=new_thread_id,
    content="Absolutely! NYC has incredible dining options. For Italian, try Carbone in Greenwich Village. For sushi, Nobu in Tribeca is world-class. For a classic NYC experience, Katz's Delicatessen on the Lower East Side is a must.",
)
memory.add_cosmos(
    user_id="user-002", role="user", thread_id=new_thread_id,
    content="I love Italian food. Are there any options that are budget-friendly?",
)
memory.add_cosmos(
    user_id="user-002", role="agent", thread_id=new_thread_id,
    content="For budget-friendly Italian in NYC, check out L'industrie Pizzeria in Williamsburg or Artichoke Basille's Pizza. Both are highly rated and won't break the bank.",
)

# Verify the memories were added directly to Cosmos DB (not in local memory)
print(f"Local memory count (should be unchanged): {len(memory.local_memory)}\n")

cosmos_results = memory.get_memories(user_id="user-002", thread_id=new_thread_id)
print(f"Memories in Cosmos DB for new thread: {len(cosmos_results)}")
for r in cosmos_results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} {r['content'][:60]}")

New Thread ID: bb7ca0ed-e4e6-4152-911d-57ce21ed751b

Local memory count (should be unchanged): 19

Memories in Cosmos DB for new thread: 5
  [bb7ca0ed...] [8f712520...] role=user   Can you recommend some good restaurants in New York City?
  [bb7ca0ed...] [52eb7686...] role=tool   {"query": "top restaurants NYC", "results": ["Carbone", "Nob
  [bb7ca0ed...] [dfea17c8...] role=agent  Absolutely! NYC has incredible dining options. For Italian, 
  [bb7ca0ed...] [573b3124...] role=user   I love Italian food. Are there any options that are budget-f
  [bb7ca0ed...] [81c3abc7...] role=agent  For budget-friendly Italian in NYC, check out L'industrie Pi


### 3c. Retrieve memories from Cosmos DB with `get_memories`

Supports the same filters as `get_local`: `memory_id`, `user_id`, `role`, `memory_type`.

In [11]:
# Get all memories for user-001
results = memory.get_memories(user_id=USER_ID)
print(f"Memories for user-001: {len(results)}\n")
for r in results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

# Get only agent memories
agent_results = memory.get_memories(role="agent")
print(f"\nAgent memories: {len(agent_results)}")
for r in agent_results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

Memories for user-001: 19

  [e14d2b93...] [a54fd759...] role=user   type=turn     What's the weather like in Seattle this weekend?
  [e14d2b93...] [34f80709...] role=agent  type=turn     This weekend Seattle will be around 55°F with part
  [e14d2b93...] [8a47c8f7...] role=agent  type=turn     Sure! I found round-trip flights departing Friday 
  [e14d2b93...] [131015e6...] role=user   type=turn     Something near Pike Place Market would be great. K
  [e14d2b93...] [f01ae2bb...] role=agent  type=turn     I found a round-trip on Alaska Airlines for $275 a
  [e14d2b93...] [fd18c1e3...] role=user   type=turn     Whenever you book a flight for me, always book an 
  [e14d2b93...] [b47560e8...] role=agent  type=turn     Got it. I'll always select an aisle seat for your 
  [e14d2b93...] [a1c02435...] role=user   type=turn     For trip planning, my workflow is: first check the
  [e14d2b93...] [fe6666fd...] role=agent  type=turn     Noted — I'll follow that order: weather, then flig
  [e14d2b93.


Agent memories: 331
  [26f28370...] [3f04a33a...] role=agent  type=turn     The PNW has amazing trails like the Wonderland Tra
  [26f28370...] [afebb21e...] role=agent  type=turn     Running is a great way to stay fit! Do you prefer 
  [26f28370...] [9b9a4a3b...] role=agent  type=turn     Python is very popular for AI/ML workloads. What f
  [96f8868c...] [afbb20b5...] role=agent  type=turn     Got it! You're based in Seattle working at Microso
  [96f8868c...] [c175440a...] role=agent  type=turn     8 years of Python experience is impressive!
  [96f8868c...] [ba7a7168...] role=agent  type=turn     That's a great area! LLMs combined with RAG can un
  [5827e67c...] [d3497550...] role=agent  type=turn     Spring is a wonderful time to visit Japan! Cherry 
  [5827e67c...] [3edcb648...] role=agent  type=turn     10–14 days lets you spend ~5 days in each city plu
  [5827e67c...] [679afa68...] role=agent  type=turn     Japan has wonderful vegetarian options — try shoji
  [5827e67c...] [44bc09

### 3d. Update & Delete in Cosmos DB

`update_cosmos` and `delete_cosmos` work just like their local counterparts. If the content changes, the embedding is automatically re-generated.

In [12]:
# Update the user's budget message to add a hotel budget constraint
user_msgs = memory.get_memories(user_id=USER_ID, role="user")
target = [m for m in user_msgs if "Pike Place" in m["content"]][0]
print(f"Before: {target['content']}\n")

memory.update_cosmos(
    memory_id=target["id"],
    content="Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
    metadata={"edited": True, "reason": "user clarified hotel budget"},
)

updated = memory.get_memories(memory_id=target["id"])[0]
print(f"After:  {updated['content']}")

Before: Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.



After:  Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.


In [13]:
# Delete the tool memory from Cosmos
tool_mems = memory.get_memories(user_id="user-002", role="tool")
print(tool_mems[0])
if tool_mems:
    memory.delete_cosmos(
        tool_mems[0]["id"],
        thread_id=tool_mems[0]["thread_id"],
        user_id=tool_mems[0]["user_id"],
    )
    print(f"Deleted tool memory {tool_mems[0]['id'][:8]}...")

# Verify
remaining = memory.get_memories()
print(f"\nRemaining memories in Cosmos DB: {len(remaining)}")
for r in remaining:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

{'id': '52eb7686-6c58-4c0f-bfd4-2a5a3579e451', 'user_id': 'user-002', 'thread_id': 'bb7ca0ed-e4e6-4152-911d-57ce21ed751b', 'role': 'tool', 'type': 'turn', 'content': '{"query": "top restaurants NYC", "results": ["Carbone", "Nobu", "Katz\'s Deli", "Le Bernardin"]}', 'metadata': {'tool_name': 'restaurant_search', 'tool_call_id': 'call_abc123'}, 'created_at': '2026-05-04T20:26:46.514321+00:00', 'tags': [], '_rid': 'EShyAMqZjm9qMjEBAAAAAA==', '_self': 'dbs/EShyAA==/colls/EShyAMqZjm8=/docs/EShyAMqZjm9qMjEBAAAAAA==/', '_etag': '"00009606-0000-0800-0000-69f901060000"', '_attachments': 'attachments/', '_ts': 1777926406}
Deleted tool memory 52eb7686...



Remaining memories in Cosmos DB: 1061
  [9060aacb...] [fact_31f...] role=system type=fact     The user is building a recommendation engine for a
  [9060aacb...] [fact_24c...] role=system type=fact     The user chose a hybrid recommendation approach co
  [9060aacb...] [fact_801...] role=system type=fact     The user plans to use embeddings generated from bo
  [9060aacb...] [fact_9e2...] role=system type=fact     The user prefers Python for development.
  [9060aacb...] [fact_d31...] role=system type=fact     The user wants to use FastAPI for the API layer of
  [9060aacb...] [ep_63631...] role=system type=episodic Last quarter, the user implemented a similar recom
  [9060aacb...] [summary_...] role=system type=summary  The user discussed building a hybrid recommendatio
  [26f28370...] [050930c1...] role=user   type=turn     I love hiking in the Pacific Northwest
  [26f28370...] [3f04a33a...] role=agent  type=turn     The PNW has amazing trails like the Wonderland Tra
  [26f28370...] [128

### 3e. Retrieve a full thread with `get_thread`

`get_thread` fetches all memories for a given `thread_id` from Cosmos DB, returned in chronological order (oldest first).

- **`thread_id`** (required) – the thread to retrieve
- **`user_id`** (optional) – filter to a specific user
- **`recent_k`** (optional) – return only the *k* most recent documents; omit to get all

In [14]:
# Use the CURRENT seed thread (from cell 5) so we operate on the data we just wrote.
thread_id = THREAD_ID
print(f"Using thread_id: {thread_id}\n")

# Get all documents in the thread
thread_all = memory.get_thread(thread_id=thread_id)
print(f"All memories in thread: {len(thread_all)}")
for m in thread_all:
    print(f"  [{m['thread_id'][:8]}...] [{m['id'][:8]}...] role={m.get('role','?'):6} type={m['type']:8}  {m['content'][:60]}")

# Most recent 2
recent = memory.get_thread(thread_id=thread_id, recent_k=2)
print(f"\nMost recent 2 memories:")
for m in recent:
    print(f"  [{m['thread_id'][:8]}...] [{m['id'][:8]}...] role={m.get('role','?'):6} type={m['type']:8}  {m['content'][:60]}")

# Filter to a specific user
thread_user = memory.get_thread(thread_id=thread_id, user_id=USER_ID)
print(f"\nThread memories for user-001: {len(thread_user)}")


Using thread_id: e14d2b93-dbdc-46e7-99db-d3c916198047

All memories in thread: 11
  [e14d2b93...] [a54fd759...] role=user   type=turn      What's the weather like in Seattle this weekend?
  [e14d2b93...] [34f80709...] role=agent  type=turn      This weekend Seattle will be around 55°F with partly cloudy 
  [e14d2b93...] [8a47c8f7...] role=agent  type=turn      Sure! I found round-trip flights departing Friday evening an
  [e14d2b93...] [131015e6...] role=user   type=turn      Something near Pike Place Market would be great. Keep flight
  [e14d2b93...] [f01ae2bb...] role=agent  type=turn      I found a round-trip on Alaska Airlines for $275 and two hot
  [e14d2b93...] [fd18c1e3...] role=user   type=turn      Whenever you book a flight for me, always book an aisle seat
  [e14d2b93...] [b47560e8...] role=agent  type=turn      Got it. I'll always select an aisle seat for your bookings.
  [e14d2b93...] [a1c02435...] role=user   type=turn      For trip planning, my workflow is: first check t

## 4. Thread Summary (in-process)

`generate_thread_summary()` runs the summarisation pipeline **in-process** — no Azure Functions
required. It will:

1. Query Cosmos DB for memories matching the given `user_id` + `thread_id`.
2. Optionally keep only the **k most recent** documents (`recent_k`).
3. Call the configured AI Foundry LLM to produce a structured summary.
4. Generate a vector embedding of the summary.
5. Upsert the result back into Cosmos DB as a memory of type `"summary"`.

If a prior summary exists, the call performs an **incremental update** that preserves the
metadata-tracked `source_count`.


In [15]:
# Use the CURRENT seed thread (from cell 5) so summarisation operates on the new data.
thread_id = THREAD_ID
user_id = USER_ID
print(f"Summarizing thread_id={thread_id}  user_id={user_id}\n")

# Run the thread summary pipeline (in-process). Returns the persisted summary doc.
summary_doc = memory.generate_thread_summary(user_id=user_id, thread_id=thread_id)


Summarizing thread_id=e14d2b93-dbdc-46e7-99db-d3c916198047  user_id=user-09577899



LLM model=gpt-5.2-chat rejected 'temperature'; retrying without it.


In [16]:
print("Summary document:")
print(f"  id: {summary_doc['id']}")
print(f"  content: {summary_doc['content'][:200]}...")
print(f"  source_count: {summary_doc.get('metadata', {}).get('source_count')}")
print(f"  embedding length: {len(summary_doc.get('embedding', []))}")


Summary document:
  id: summary_user-09577899_e14d2b93-dbdc-46e7-99db-d3c916198047
  content: The conversation focused on planning a weekend trip to Seattle, including checking the weather, identifying flights and hotels near Pike Place Market within a specified budget, and establishing the us...
  source_count: 11
  embedding length: 1536


## 5. Memory Extraction (facts + episodic + procedural)

`extract_memories()` runs a single LLM call that produces three structured memory types:

| Type        | Description                                                              |
|-------------|--------------------------------------------------------------------------|
| `fact`      | Stable, atomic facts about the user.                                     |
| `episodic`  | Discrete past events ("Last spring, the user hiked Mount Rainier.").     |
| `procedural`| How-to guidance the agent should follow ("When debugging the LLM, ...").|

Each derived memory is embedded and stored in Cosmos DB with auto-generated tags and a
salience score. The call returns a count summary.


In [17]:
# Extract memories from the SEED thread (mixed factual + booking specifics).
result_main = memory.extract_memories(user_id=USER_ID, thread_id=THREAD_ID)
print("Seed thread extraction:", result_main)

# Extract from the pure-procedural RULES thread to surface procedural memories.
result_rules = memory.extract_memories(user_id=USER_ID, thread_id=RULES_THREAD_ID)
print("Rules thread extraction:", result_rules)


LLM model=gpt-5.2-chat rejected 'temperature'; retrying without it.


Seed thread extraction: {'facts_count': 0, 'procedural_count': 0, 'episodic_count': 0, 'updated_count': 0}


LLM model=gpt-5.2-chat rejected 'temperature'; retrying without it.


Rules thread extraction: {'facts_count': 0, 'procedural_count': 0, 'episodic_count': 0, 'updated_count': 0}


In [18]:
# Inspect the persisted derived memories across BOTH threads for this user.
for mt in ("fact", "episodic", "procedural"):
    docs = memory.get_memories(user_id=USER_ID, memory_type=mt)
    print(f"\n{mt.upper()}S ({len(docs)}):")
    for d in docs:
        print(f"  • {d['content'][:100]}  [salience={d.get('salience')}]")


get_memories returned empty results



FACTS (3):
  • The user prefers hotels located near Pike Place Market in Seattle.  [salience=0.7]
  • The user requires round-trip flights to cost under $300.  [salience=0.9]
  • The user requires hotel stays to cost under $200 per night.  [salience=0.9]

EPISODICS (0):

PROCEDURALS (9):
  • Always book an aisle seat when booking flights for the user; never select a window or middle seat.  [salience=0.9]
  • Follow this order for trip planning: first check the weather, then check flights, and book the hotel  [salience=0.85]
  • Never book travel that departs or arrives between midnight and 6am unless the user explicitly approv  [salience=0.9]
  • Only recommend hotels that include complimentary breakfast.  [salience=0.85]
  • Always book an aisle seat and never select a window or middle seat when booking flights for the user  [salience=0.85]
  • Follow this order for trip planning: first check the weather for the destination, then check flights  [salience=0.8]
  • Never book flights t

## 6. User Summary (cross-thread profile)

`generate_user_summary()` aggregates memories **across all threads** for a user and produces
a structured profile (preferences, account state, behavioural patterns, …). The result is
stored in Cosmos DB with `type: "user_summary"`.

- **`user_id`** (required) – the user to build the profile for
- **`thread_ids`** (optional) – limit to specific threads; omit for all threads
- **`recent_k`** (optional) – per-thread recency limit

Retrieve the latest stored profile at any time with `get_user_summary(user_id)` — useful
for priming new conversations.


In [19]:
# Generate a user-level summary across all threads for user_id.
user_summary_doc = memory.generate_user_summary(user_id=user_id)
print("User summary id:", user_summary_doc["id"])
print("\nContent:\n", user_summary_doc["content"][:600], "...")
print("\nStructured profile keys:", list(user_summary_doc.get("metadata", {}).get("structured_summary", {}).keys()))


LLM model=gpt-5.2-chat rejected 'temperature'; retrying without it.


User summary id: user_summary_user-09577899

Content:
 {
  "key_facts": [],
  "personal_preferences": [
    "The user requires aisle seats for all flight bookings and explicitly does not want window or middle seats.",
    "The user does not want flights that depart or arrive between midnight and 6am unless they explicitly approve it.",
    "The user prefers hotels that include complimentary breakfast and only wants such hotels recommended.",
    "The user prefers travel planning to follow a specific order: check the weather first, then check flights, and book the hotel last after everything else is confirmed.",
    "The user sets explicit budget c ...

Structured profile keys: ['key_facts', 'personal_preferences', 'account_environment', 'goals_current_work', 'behavioral_patterns', 'compliance_requirements', 'open_items', 'topics']


In [20]:
# Retrieve the stored user summary from Cosmos DB
stored = memory.get_user_summary(user_id=user_id)
if stored:
    print("User Summary for", user_id)
    print(stored[0]["content"])
else:
    print("No user summary found — run the generate_user_summary cell first.")


User Summary for user-09577899
{
  "key_facts": [],
  "personal_preferences": [
    "The user requires aisle seats for all flight bookings and explicitly does not want window or middle seats.",
    "The user does not want flights that depart or arrive between midnight and 6am unless they explicitly approve it.",
    "The user prefers hotels that include complimentary breakfast and only wants such hotels recommended.",
    "The user prefers travel planning to follow a specific order: check the weather first, then check flights, and book the hotel last after everything else is confirmed.",
    "The user sets explicit budget constraints for travel (e.g., under $300 round trip for flights and under $200 per night for hotels in one example)."
  ],
  "account_environment": [],
  "goals_current_work": [
    "The user is planning trips that involve booking flights and hotels with defined constraints and preferences.",
    "The user is planning a weekend trip to Seattle, seeking flights under $

### 7. Vector search with `search_cosmos`

`search_cosmos` embeds a natural-language query via your AI Foundry model and runs a `VectorDistance` similarity search in Cosmos DB. Optionally filter by `user_id` or `thread_id`.


In [21]:
results = memory.search_cosmos(
    search_terms="What did the user ask about the weather?",
    user_id=USER_ID,
    top_k=3,
)

In [22]:
print(f"Top {len(results)} results:\n")
for r in results:
    score = r.get("score", "N/A")
    print(f"  [{r['id'][:8]}...] score={score}  {r['content'][:60]}")

Top 3 results:

  [summary_...] score=N/A  The conversation focused on planning a weekend trip to Seatt
  [user_sum...] score=N/A  {
  "key_facts": [],
  "personal_preferences": [
    "The us
  [proc_6ec...] score=N/A  Follow this trip planning order: first check the destination
